In [4]:
# Exemple simple montrant l'intérêt du *

# Sans * - confusion possible avec positional
def transfer_credits_bad(user, amount, reason="payment"):
    return f"{user} reçoit {amount}€ pour: {reason}"

# Appel ambigu - lequel est le montant?
result = transfer_credits_bad("Alice", 100, "bonus")  # OK, mais peu clair

# Avec * - "reason" DOIT être keyword-only (plus sûr)
def transfer_credits_safe(user, amount, *, reason="payment"): # Faire le test en supprimant le *
    return f"{user} reçoit {amount}€ pour: {reason}"

# Appel clair et explicite
result = transfer_credits_safe("Alice", 100, reason="bonus")

# Ceci échoue - protège contre les erreurs
transfer_credits_safe("Alice", 100, "bonus")  # Provoque TypeError!

TypeError: transfer_credits_safe() takes 2 positional arguments but 3 were given

In [5]:
# Exemple simple montrant l'intérêt de nonlocal

# Sans nonlocal - la variable n'est pas modifiée
def counter_bad():
    count = 0
    
    def increment():
        count = count + 1  # Crée une NOUVELLE variable locale
        return count
    
    increment()
    increment()
    return count  # Toujours 0!

print(counter_bad())  # Affiche 0 - les incréments n'ont pas d'effet

# Avec nonlocal - modification de la variable de l'enclosing scope
def counter_good():
    count = 0
    
    def increment():
        nonlocal count  # Accède à la variable de l'enclosing scope
        count = count + 1
        return count
    
    increment()
    increment()
    return count  # Maintenant 2!

print(counter_good())  # Affiche 2 - les incréments fonctionnent

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

In [15]:
# Exemple simple montrant l'intérêt des lambdas

# Sans lambda - fonction traditionnelle
def double(x):
    return x * 2

result = double(5)
print(result)  # Affiche 10

# Avec lambda - syntaxe courte et inline
double_lambda = lambda x: x * 2

result = double_lambda(5)
print(result)  # Affiche 10

# Avantage réel: avec map, filter, sorted
numbers = [1, 2, 3, 4, 5]

# Sans lambda - plus verbeux
def is_even(n):
    return n % 2 == 0

evens = list(filter(is_even, numbers))
print(evens)  # [2, 4]

# Avec lambda - concis et lisible
evens = list(filter(lambda n: n % 2 == 0, numbers))
print(evens)  # [2, 4]

# Tri avec lambda
students = [("Alice", 18), ("Bob", 16), ("Charlie", 17)]
sorted_by_age = sorted(students, key=lambda s: s[1])
print(sorted_by_age)  # [("Bob", 16), ("Charlie", 17), ("Alice", 18)]

10
10
[2, 4]
[2, 4]
[('Bob', 16), ('Charlie', 17), ('Alice', 18)]


In [10]:
# Exemple ultra-simple des décorateurs

import time
from functools import wraps

# Décorateur simple - ajoute du timing
def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"⏱️  {func.__name__} a pris {elapsed:.2f}s")
        return result
    return wrapper

# Sans décorateur - code mélangé
def addition_bad():
    time.sleep(0.5)
    return 2 + 2

# Avec décorateur - code propre
@timer
def addition_good():
    time.sleep(0.5)
    return 2 + 2

addition_good()  # Affiche: ⏱️  addition_good a pris 0.50s

@timer
def multiplication():
    time.sleep(0.2)
    return 3 * 4

multiplication()  # Affiche: ⏱️  multiplication a pris 0.20s

⏱️  addition_good a pris 0.51s
⏱️  multiplication a pris 0.20s


12

In [16]:
# Exemple ultra-simple : créer des comptes bancaires

# ❌ SANS CLASS - Données éparses et confuses
def create_account_bad(owner):
    return {"owner": owner, "balance": 0}

def deposit_bad(account, amount):
    account["balance"] += amount
    return account

# Problème : les données peuvent se corrompre facilement
alice_account = create_account_bad("Alice")
alice_account["balance"] = -1000  # ✗ Solde négatif ? Rien ne l'empêche !
alice_account["random_field"] = "oups"  # ✗ On ajoute n'importe quoi

print(alice_account)  # {'owner': 'Alice', 'balance': -1000, 'random_field': 'oups'}


# ✅ AVEC CLASS - Données protégées et cohérentes
class BankAccount:
    def __init__(self, owner):
        self.owner = owner
        self.balance = 0
    
    def deposit(self, amount):
        self.balance += amount
        return f"{self.owner} a déposé {amount}€ (solde: {self.balance}€)"
    
    def withdraw(self, amount):
        if amount > self.balance:
            return f"Solde insuffisant ! (solde: {self.balance}€)"
        self.balance -= amount
        return f"{self.owner} a retiré {amount}€ (solde: {self.balance}€)"

# Utilisation simple et sûre
alice = BankAccount("Alice")
print(alice.deposit(500))      # Alice a déposé 500€ (solde: 500€)
print(alice.deposit(200))      # Alice a déposé 200€ (solde: 700€)
print(alice.withdraw(300))     # Alice a retiré 300€ (solde: 400€)
print(alice.withdraw(1000))    # Solde insuffisant ! (solde: 400€)

# Avec la class, on ne peut pas créer de solde négatif par erreur !

{'owner': 'Alice', 'balance': -1000, 'random_field': 'oups'}
Alice a déposé 500€ (solde: 500€)
Alice a déposé 200€ (solde: 700€)
Alice a retiré 300€ (solde: 400€)
Solde insuffisant ! (solde: 400€)


In [13]:
# Exemple simple du polymorphisme

# Le polymorphisme : la même méthode, des comportements différents

class Chat:
    def faire_bruit(self):
        return "Miaou ! 🐱"

class Chien:
    def faire_bruit(self):
        return "Ouaf ! 🐶"

class Canard:
    def faire_bruit(self):
        return "Coin coin ! 🦆"

# Polymorphisme : même appel, résultats différents
animaux = [Chat(), Chien(), Canard()]

print("=== Sans polymorphisme (répétitif) ===")
chat = Chat()
print(chat.faire_bruit())

chien = Chien()
print(chien.faire_bruit())

canard = Canard()
print(canard.faire_bruit())

print("\n=== Avec polymorphisme (magique) ===")
for animal in animaux:
    print(animal.faire_bruit())  # Même appel, comportements différents !

=== Sans polymorphisme (répétitif) ===
Miaou ! 🐱
Ouaf ! 🐶
Coin coin ! 🦆

=== Avec polymorphisme (magique) ===
Miaou ! 🐱
Ouaf ! 🐶
Coin coin ! 🦆


In [14]:
# Exemple simple : HÉRITAGE vs COMPOSITION

# ===== HÉRITAGE =====
# Une classe "enfant" hérite des propriétés et méthodes d'une classe "parent"

class Animal:
    """Classe parent - contient ce que tous les animaux ont en commun"""
    def __init__(self, name):
        self.name = name
    
    def faire_bruit(self):
        return "Son générique"

class Chien(Animal):
    """Classe enfant - hérite de Animal"""
    def faire_bruit(self):
        return f"{self.name} dit: Ouaf ! 🐶"

class Chat(Animal):
    """Classe enfant - hérite de Animal"""
    def faire_bruit(self):
        return f"{self.name} dit: Miaou ! 🐱"

# Utilisation de l'héritage
rex = Chien("Rex")
minou = Chat("Minou")

print("=== HÉRITAGE ===")
print(rex.faire_bruit())      # Rex dit: Ouaf ! 🐶
print(minou.faire_bruit())    # Minou dit: Miaou ! 🐱


# ===== COMPOSITION =====
# Un objet est "composé" d'autres objets plutôt que d'en hériter

class Moteur:
    """Composant indépendant"""
    def demarrer(self):
        return "Vroom ! 🔧"

class Voiture:
    """Une voiture est composée d'un moteur (elle contient un moteur)"""
    def __init__(self, brand, moteur):
        self.brand = brand
        self.moteur = moteur  # La voiture HAS-A un moteur
    
    def demarrer(self):
        return f"{self.brand} démarre: {self.moteur.demarrer()}"

# Utilisation de la composition
moteur_diesel = Moteur()
ma_voiture = Voiture("Peugeot", moteur_diesel)

print("\n=== COMPOSITION ===")
print(ma_voiture.demarrer())  # Peugeot démarre: Vroom ! 🔧


# ===== RÉSUMÉ =====
# HÉRITAGE: Chien IS-A Animal (un chien est un animal)
# COMPOSITION: Voiture HAS-A Moteur (une voiture a un moteur)

=== HÉRITAGE ===
Rex dit: Ouaf ! 🐶
Minou dit: Miaou ! 🐱

=== COMPOSITION ===
Peugeot démarre: Vroom ! 🔧
